# NDT7 (M-Lab) Data Prep — Thailand Broadband + Mobile, Province x Quarter

Aggregates `../../../data/ndt7/th/mlab_th_clean.parquet` (60.2M raw NDT7 test records, already ISP-classified and province-joined via
per-IP lookup + point-in-polygon) into province x quarter format, split into Broadband and
Mobile/Cellular parts, mirroring the same structure across all three NDT7 "tigger" countries
(Cambodia/Thailand/Vietnam).

**Rebuilt to use DuckDB instead of a manual pyarrow-batch-streaming loop** — DuckDB reads the
parquet file directly and does the tile-binning + GROUP BY aggregation out-of-core (no manual
batching code needed, no risk of the memory issues the streaming version was written to avoid).
The tile-binning and weighted-aggregation formulas are byte-for-byte unchanged from the pandas
version — verified against the prior pandas-based export (float-precision-only differences,
~1e-13, from AVG() accumulation order).

No province-name mapping needed — Thailand's raw `province` values already match `province_reference.csv`.

Same tile scheme as Ookla's own published tiles (zoom-16 slippy tiles, ~610m) — keeps
`n_tiles`/`is_reliable` comparable across Ookla and NDT7, and across countries:
`total_tests >= 100` only (n_tiles dropped for NDT7 — MaxMind gives city-centroid coordinates, so n_tiles measures cities-per-province, not data spread; Ookla keeps both).

**Outputs:**
- `data/exports/ndt7_thailand_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_thailand_province_quarterly.csv` — Mobile/Cellular
  (renamed from `ndt7_thailand_mobile_...` to match Ookla's `ookla_mobile_<country>_...`
  naming convention — position of "mobile" now matches across both pipelines)

In [1]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/th/mlab_th_clean.parquet'
TH_REF_CSV = '../../../data/reference/province_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3

### 1. Tile-Binning + Province-Quarter Aggregation (DuckDB)

All heavy row-level work (filtering, quarter-labeling, zoom-16 mercator tile assignment, GROUP BY tile x quarter x type x network_type) happens in one DuckDB SQL query against the raw parquet — no Python-side batching.

In [2]:
con = duckdb.connect()
con.execute("SET memory_limit='10GB'")                        # PH/ID ใหญ่ ต้องตั้ง
con.execute("SET temp_directory='../../../.tmp/duckdb'")   # ที่พักตอน spill
con.execute("SET preserve_insertion_order=false")

sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps,
        min_rtt,
        type, network_type, province,
        year,
        CAST(CEIL(month / 3.0) AS INT) AS qtr,
        -- zoom-16 Web Mercator tile — ใช้เป็นคอลัมน์วินิจฉัยเท่านั้น ไม่ได้ใช้คิดค่าเฉลี่ย
        CAST(FLOOR((longitude + 180) / 360 * 65536) AS BIGINT) AS tx,
        CAST(FLOOR((1 - (ln(tan(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878)))
             + 1.0/cos(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878))))) / pi()) / 2 * 65536) AS BIGINT) AS ty
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND province IS NOT NULL
      AND network_type IN ('broadband', 'cellular')
)
SELECT
    province, network_type, type,
    (CAST(year AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
    AVG(mean_throughput_mbps)                       AS avg_thr,
    AVG(CASE WHEN min_rtt < 2000 THEN min_rtt END)  AS avg_lat,
    COUNT(*)                                        AS test_count,
    COUNT(DISTINCT CAST(tx AS VARCHAR) || '_' || CAST(ty AS VARCHAR)) AS n_tiles
FROM filtered
GROUP BY province, network_type, type, year_q
"""

tile_agg_all = con.execute(sql).df()
print(f"province x quarter x type x network rows: {len(tile_agg_all):,}")
print(f"quarters: {len(tile_agg_all['year_q'].unique())} | province: {tile_agg_all['province'].nunique()}")
print(tile_agg_all.groupby('network_type')['test_count'].sum().apply(lambda x: f'{x:,}'))

province x quarter x type x network rows: 2,785
quarters: 12 | province: 77
network_type
broadband    34,122,640
cellular     24,405,235
Name: test_count, dtype: object


### 3. Province-Level Weighted Aggregation (per network type)

In [3]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    d = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] province x quarter x type rows: {len(d):,}")

    dl = d[d['type'] == 'download'].rename(columns={
        'avg_thr': 'avg_d_mbps', 'avg_lat': 'avg_lat_ms_wt', 'test_count': 'total_tests'})
    ul = d[d['type'] == 'upload'].rename(columns={'avg_thr': 'avg_u_mbps'})

    dl_stats = dl[['year_q', 'province', 'avg_d_mbps', 'avg_lat_ms_wt', 'total_tests', 'n_tiles']]
    ul_stats = ul[['year_q', 'province', 'avg_u_mbps']]

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    # NDT7 ใช้ total_tests อย่างเดียว ไม่ใช้ n_tiles เป็นเกณฑ์ (Ookla ยังใช้ทั้งคู่)
    # เหตุผล: NDT7 ได้พิกัดจาก MaxMind ซึ่งเป็น city centroid ทุก test ในเมืองเดียวกันจึงตกลง tile
    # เดียวกัน n_tiles จึงวัด "จังหวัดนี้มีกี่เมืองใน MaxMind" ไม่ได้วัดการกระจายตัวของข้อมูล
    # (ลาวทั้งประเทศมีพิกัดต่างกัน 33 จุด n_tiles สูงสุด = 3 -> เกณฑ์ >=5 เป็นไปไม่ได้)
    # คอลัมน์ n_tiles ยังเก็บไว้ให้ดูใน "Data Quality" ของ EDA
    master['is_reliable'] = master['total_tests'] >= 100
    print(f"[{network_type}] province x quarter rows: {len(master)} | "
          f"reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

In [4]:
ref = pd.read_csv(TH_REF_CSV)

---
## Part 1 — Broadband

In [5]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] province x quarter x type rows: 1,804
[broadband] province x quarter rows: 902 | reliable: 872 (96.7%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Amnat Charoen,64.426268,160.035423,822,5,35.014625,2023,1,True,Northeastern,4,372000,77048,113,7939.27,253897.82
1,2023-Q1,Ang Thong,75.611123,127.253152,2827,7,47.997056,2023,1,True,Central,3,269000,142287,283,14661.70,468881.20
2,2023-Q1,Bangkok Metropolis,88.934056,100.929037,811120,49,58.772441,2023,1,True,Bangkok & Vicinity,1,5456000,593927,3488,61200.11,1957179.52
3,2023-Q1,Bueng Kan,52.880631,182.764677,373,5,33.569460,2023,1,True,Northeastern,4,419000,80159,105,8259.84,264149.56
4,2023-Q1,Buri Ram,65.528579,139.445712,3982,14,41.378571,2023,1,True,Northeastern,4,1566000,80684,155,8313.93,265879.60


In [6]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_thailand_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 902 rows -> ../../../data/exports/ndt7_thailand_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Amnat Charoen,2023-Q1,2023,1,64.426268,35.014625,160.035423,822,5,True,Northeastern,4,372000,77048,113,7939.27,253897.82
1,Ang Thong,2023-Q1,2023,1,75.611123,47.997056,127.253152,2827,7,True,Central,3,269000,142287,283,14661.70,468881.20
2,Bangkok Metropolis,2023-Q1,2023,1,88.934056,58.772441,100.929037,811120,49,True,Bangkok & Vicinity,1,5456000,593927,3488,61200.11,1957179.52


---
## Part 2 — Mobile/Cellular

In [7]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] province x quarter x type rows: 981
[cellular] province x quarter rows: 494 | reliable: 354 (71.7%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Ang Thong,8.604534,85.327286,7.0,2.0,3.035360,2023,1,False,Central,3,269000,142287,283,14661.70,468881.20
1,2023-Q1,Bangkok Metropolis,19.433952,86.666889,1104039.0,39.0,8.510880,2023,1,True,Bangkok & Vicinity,1,5456000,593927,3488,61200.11,1957179.52
2,2023-Q1,Chachoengsao,87.777752,86.787533,230.0,3.0,71.765227,2023,1,True,Eastern,2,733000,400385,142,41256.93,1319396.70
3,2023-Q1,Chai Nat,7.557913,89.284943,71.0,2.0,3.986460,2023,1,False,Central,3,314000,135667,131,13979.56,447066.18
4,2023-Q1,Chaiyaphum,10.801072,61.150368,87.0,1.0,4.139980,2023,1,False,Northeastern,4,1106000,73134,88,7535.96,240999.93


In [8]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_thailand_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 494 rows -> ../../../data/exports/ndt7_mobile_thailand_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Ang Thong,2023-Q1,2023,1,8.604534,3.035360,85.327286,7.0,2.0,False,Central,3,269000,142287,283,14661.70,468881.20
1,Bangkok Metropolis,2023-Q1,2023,1,19.433952,8.510880,86.666889,1104039.0,39.0,True,Bangkok & Vicinity,1,5456000,593927,3488,61200.11,1957179.52
2,Chachoengsao,2023-Q1,2023,1,87.777752,71.765227,86.787533,230.0,3.0,True,Eastern,2,733000,400385,142,41256.93,1319396.70


## Summary

- Input: Thailand NDT7 raw test records, already province-joined + ISP-classified
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook and
  the other NDT7 "tigger" prep notebooks
- Engine: DuckDB (was: manual pyarrow-batch-streaming loop in pandas) — verified to reproduce
  the prior pandas-based export exactly (float-precision-only differences)